* Trend **** Fill all the null values in train_clinical_all with median and generate the data, removing the month >= 84 as there are too many null values.****

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

import amp_pd_peptide

from scipy.optimize import minimize

In [ ]:
def smape_plus_1(y_true, y_pred):
    """
    Calculates the Symmetric Mean Absolute Percentage Error (SMAPE) with an offset of 1.

    Parameters:
    - y_true : Array or list of true values.
    - y_pred : Array or list of predicted values.

    Returns:
    - float: SMAPE value as a percentage.
    """
    y_true_plus_1 = y_true + 1
    y_pred_plus_1 = y_pred + 1
    metric = np.zeros(len(y_true_plus_1))
    
    numerator = np.abs(y_true_plus_1 - y_pred_plus_1)
    denominator = ((np.abs(y_true_plus_1) + np.abs(y_pred_plus_1)) / 2)
    
    mask_not_zeros = (y_true_plus_1 != 0) | (y_pred_plus_1 != 0)
    metric[mask_not_zeros] = numerator[mask_not_zeros] / denominator[mask_not_zeros]
    
    return 100 * np.nanmean(metric)

In [ ]:
# Read the clinical data from the file '/kaggle/input/amp-parkinsons-disease-progression-prediction/train_clinical_data.csv'
train_clinical_data = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/train_clinical_data.csv')

# Assign the value 'standard' to the 'source' column in the 'train_clinical_data' DataFrame
train_clinical_data['source'] = 'standard'

# Read the supplemental clinical data from the file '/kaggle/input/amp-parkinsons-disease-progression-prediction/supplemental_clinical_data.csv'
supplemental_clinical_data = pd.read_csv('/kaggle/input/amp-parkinsons-disease-progression-prediction/supplemental_clinical_data.csv')

# Assign the value 'supplemental' to the 'source' column in the 'supplemental_clinical_data' DataFrame
supplemental_clinical_data['source'] = 'supplemental'

# Concatenate the 'train_clinical_data' and 'supplemental_clinical_data' DataFrames vertically to create 'train_clinical_all'
train_clinical_all = pd.concat([train_clinical_data, supplemental_clinical_data])

# Return the combined DataFrame 'train_clinical_all'
train_clinical_all

In [ ]:
# fill up empty values
# Group by visit_month and calculate the median of updrs1~4 columns for each month
medians = train_clinical_all.groupby('visit_month')[['updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']].median()
medians

In [ ]:
merged_df = pd.merge(train_clinical_all, medians, on='visit_month', suffixes=('_df1', '_df2'))
merged_df

In [ ]:
# fill null values with median values and drop unnecessary columns
merged_df['updrs_1'] = merged_df['updrs_1_df1'].fillna(merged_df['updrs_1_df2'])
merged_df['updrs_2'] = merged_df['updrs_2_df1'].fillna(merged_df['updrs_2_df2'])
merged_df['updrs_3'] = merged_df['updrs_3_df1'].fillna(merged_df['updrs_3_df2'])
merged_df['updrs_4'] = merged_df['updrs_4_df1'].fillna(merged_df['updrs_4_df2'])
merged_df = merged_df.drop(['updrs_1_df2', 'updrs_2_df2', 'updrs_3_df2', 'updrs_4_df2'], axis=1)
merged_df = merged_df.drop(['updrs_1_df1', 'updrs_2_df1', 'updrs_3_df1', 'updrs_4_df1'], axis=1)
merged_df

In [ ]:
# delete visit_month 3, 5, 9 (there are no such visit_months in the Test API)
train_clinical_all = merged_df[~merged_df.visit_month.isin([3, 5, 9])]
train_clinical_all

In [ ]:
train_clinical_all['pred_month'] = train_clinical_all['visit_month']

for plus_month in [6, 12, 24]:
    train_shift = train_clinical_all[['patient_id', 'visit_month', 'pred_month', 'updrs_1', 'updrs_2', 'updrs_3', 'updrs_4']].copy()
    train_shift['visit_month'] -= plus_month
    train_shift.rename(columns={f'updrs_{i}': f'updrs_{i}_plus_{plus_month}' for i in range(1, 5)}, inplace=True)
    train_shift.rename(columns={'pred_month': f'pred_month_plus_{plus_month}'}, inplace=True)
    train_clinical_all = train_clinical_all.merge(train_shift, how='left', on=['patient_id', 'visit_month'])

train_clinical_all.rename(columns={f'updrs_{i}': f'updrs_{i}_plus_0' for i in range(1, 5)}, inplace=True)
train_clinical_all.rename(columns={'pred_month': f'pred_month_plus_0'}, inplace=True)
train_clinical_all = train_clinical_all[train_clinical_all['visit_month'] <= 83]
train_clinical_all

In [ ]:
def calculate_predicitons(pred_month, trend):
#     if target == 'updrs_1': pred_month = pred_month.clip(36, None)
    if target == 'updrs_4': pred_month = pred_month.clip(36, None)
    return np.round(trend[0] + pred_month * trend[1])

def function_to_minimize(x):    
    metric = smape_plus_1(
        y_true=y_true_array, 
        y_pred=calculate_predicitons(
            pred_month=pred_month_array,
            trend=x
        )
    )
    return metric

target_to_trend = {}
for i in range(1, 5):
    target = f'updrs_{i}'
    columns_with_target = [f'{target}_plus_{plus_month}' for plus_month in [0, 6, 12, 24]]
    columns_with_pred_month = [f'pred_month_plus_{plus_month}' for plus_month in [0, 6, 12, 24]]
    y_true_array = train_clinical_all[columns_with_target].values.ravel()
    pred_month_array = train_clinical_all[columns_with_pred_month].values.ravel()
    trend = list(minimize(
        fun=function_to_minimize,
        x0=[0, 0.0048],
        method='Powell'
    ).x)
    target_to_trend[target] = trend
target_to_trend

In [ ]:
amp_pd_peptide.make_env.func_dict['__called__'] = False
env = amp_pd_peptide.make_env()   # initialize the environment
iter_test = env.iter_test()    # an iterator which loops over the test files

# The API will deliver four dataframes in this specific order:
for test_clinical_data, test_peptides, test_proteins, sample_submission in iter_test:
    sample_submission['patient_id'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[0]))
    sample_submission['visit_month'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[1]))
    sample_submission['target_name'] = sample_submission['prediction_id'].map(lambda x: 'updrs_' + x.split('_')[3])
    sample_submission['plus_month'] = sample_submission['prediction_id'].map(lambda x: int(x.split('_')[5]))
    sample_submission['pred_month'] = sample_submission['visit_month'] + sample_submission['plus_month']
    
    for i in range(1, 5):
        target = f'updrs_{i}'
        mask_target = sample_submission['target_name'] == target
        sample_submission.loc[mask_target, 'rating'] = calculate_predicitons(
            pred_month=sample_submission.loc[mask_target, 'pred_month'],
            trend=target_to_trend[target]
        )
        
    # call the env.predict for every iteration
    env.predict(sample_submission[['prediction_id', 'rating']])